# Phase 04A: KD Reproduction Entry

Notebook này khóa giao thức tái lập Knowledge Distillation và phát hành quyền
tạo notebook preflight đầu tiên cho cặp:

**Teacher A CLIP ViT-B/32 → Student S1 MobileNetV3-Large bằng response/logit KD.**

Phase 04A chưa tải checkpoint, chưa đọc ảnh/caption, chưa thực hiện backward và
không mở Held-out Test.

## Kaggle inputs bắt buộc

1. Phase 01 formal output.
2. Phase 02A shared Student pipeline.
3. Phase 03A Teacher Pool Entry.
4. Phase 03E Teacher A three-seed aggregate.
5. Phase 03H Teacher B three-seed aggregate.
6. Phase 03L Teacher C aggregate và Phase 3 COMPLETE.

## Lý do chọn cặp đầu tiên

- Teacher A và S1 dùng shared caption decoder/vocabulary nên logit KD có thể
  được định nghĩa trực tiếp theo token ID.
- S1 là Student nhỏ nhất, phù hợp mục tiêu triển khai tài nguyên hạn chế.
- Teacher A đã qua Teacher Gate ở cả ba seed.
- Teacher B vẫn được đăng ký cho sequence/feature KD ở nhánh sau; direct logit
  KD với BLIP tiếp tục bị cấm do vocabulary mismatch.
- Teacher C → S3 được đăng ký cho logit, feature và sequence KD; feature KD
  phải căn chỉnh 196 visual tokens của Teacher về 49 tokens của Student.


## Cell 1 — Environment và input discovery

### Công việc

- tạo output tree Phase 04A;
- tìm đúng Phase 1, Phase 2A, Phase 3A, Phase 3E, Phase 3H và Phase 3L;
- chỉ chấp nhận Phase 4 khi Phase 03L phát hành authorization hợp lệ;
- kiểm tra Test vẫn sealed;
- chưa đọc dữ liệu hoặc checkpoint.

### Đầu ra dự kiến

```text
PHASE 04A CELL 1 STATUS
Status                    : PASSED
Required upstream inputs  : 6 / 6 FOUND
Phase 3 Teacher Pool      : COMPLETE
Phase 4 authorization     : VERIFIED
Dataset read              : False
Checkpoint read           : False
Test Set status           : SEALED
```


In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import time
from datetime import datetime, timezone
from pathlib import Path


CELL_STARTED_AT = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//phase4_kd_reproduction_entry"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
RESULT_DIR = OUTPUT_ROOT / "results"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT,
    AUDIT_DIR,
    CONFIG_DIR,
    RESULT_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


PHASE1_READY = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
PHASE2_READY = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
PHASE3A_COMPLETE = exactly_one("PHASE2_STUDENT_BASELINES_COMPLETE.json")
TEACHER_A_COMPLETE = exactly_one(
    "PHASE3_TEACHER_A_THREE_SEED_COMPLETE.json"
)
TEACHER_B_COMPLETE = exactly_one(
    "PHASE3_TEACHER_B_THREE_SEED_AGGREGATE_COMPLETE.json"
)
PHASE3_POOL_COMPLETE = exactly_one(
    "PHASE3_TEACHER_POOL_COMPLETE.json"
)
PHASE4_AUTHORIZED = exactly_one(
    "PHASE4_KD_REPRODUCTION_AUTHORIZED.json"
)

phase3a_marker = read_json(PHASE3A_COMPLETE)
teacher_a_marker = read_json(TEACHER_A_COMPLETE)
teacher_b_marker = read_json(TEACHER_B_COMPLETE)
phase3_pool_marker = read_json(PHASE3_POOL_COMPLETE)
phase4_authorized_marker = read_json(PHASE4_AUTHORIZED)
assert phase3a_marker["test_set_status"] == "sealed"
assert phase3a_marker["test_set_used"] is False
assert teacher_a_marker["test_set_status"] == "sealed"
assert teacher_a_marker["test_set_used"] is False
assert teacher_b_marker["test_set_status"] == "sealed"
assert teacher_b_marker["test_set_used"] is False
assert phase3_pool_marker["status"] == "complete"
assert phase3_pool_marker["teacher_family_count"] == 3
assert phase3_pool_marker["teacher_seed_run_count"] == 9
assert phase3_pool_marker["all_teacher_quality_gates_passed"] is True
assert phase3_pool_marker["all_teacher_runs_qualified_for_kd"] is True
assert phase3_pool_marker["phase4_authorized"] is True
assert phase3_pool_marker["test_set_status"] == "sealed"
assert phase3_pool_marker["test_set_used"] is False
assert phase4_authorized_marker["status"] == "authorized"
assert phase4_authorized_marker["phase3_teacher_families_complete"] == 3
assert phase4_authorized_marker["phase3_teacher_seed_runs_complete"] == 9
assert phase4_authorized_marker["test_set_status"] == "sealed"
assert phase4_authorized_marker["test_set_used"] is False

phase3_authorization_ref = phase3_pool_marker["phase4_authorization"]
assert PHASE4_AUTHORIZED == (
    PHASE3_POOL_COMPLETE.parent / phase3_authorization_ref["relative_path"]
).resolve()
assert sha256_file(PHASE4_AUTHORIZED) == phase3_authorization_ref["sha256"]

write_json(
    AUDIT_DIR / "phase4a_cell01_discovery.json",
    {
        "audit_version": "phase4a_cell01_discovery_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "python": platform.python_version(),
        "inputs": {
            "phase1": str(PHASE1_READY),
            "phase2": str(PHASE2_READY),
            "phase3a": str(PHASE3A_COMPLETE),
            "teacher_a": str(TEACHER_A_COMPLETE),
            "teacher_b": str(TEACHER_B_COMPLETE),
            "phase3_pool_complete": str(PHASE3_POOL_COMPLETE),
            "phase4_authorized": str(PHASE4_AUTHORIZED),
        },
        "phase3_pool_complete_sha256": sha256_file(
            PHASE3_POOL_COMPLETE
        ),
        "phase4_authorization_sha256": sha256_file(
            PHASE4_AUTHORIZED
        ),
        "teacher_families": 3,
        "teacher_seed_runs": 9,
        "dataset_read": False,
        "caption_read": False,
        "checkpoint_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04A CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Required upstream inputs  : 6 / 6 FOUND")
print("Phase 3 Teacher Pool      : COMPLETE")
print("Teacher families          : 3 / 3 COMPLETE")
print("Teacher seed runs         : 9 / 9 COMPLETE")
print("Phase 4 authorization     : VERIFIED")
print("Dataset read              : False")
print("Caption read              : False")
print("Checkpoint read           : False")
print("Test Set status           : SEALED")
print("Output root               :", OUTPUT_ROOT)
print("Next                      : verify upstream integrity")


PHASE 04A CELL 1 STATUS
Status                    : PASSED
Required upstream inputs  : 6 / 6 FOUND
Phase 3 Teacher Pool      : COMPLETE
Teacher families          : 3 / 3 COMPLETE
Teacher seed runs         : 9 / 9 COMPLETE
Phase 4 authorization     : VERIFIED
Dataset read              : False
Caption read              : False
Checkpoint read           : False
Test Set status           : SEALED
Output root               : /kaggle/working//phase4_kd_reproduction_entry
Next                      : verify upstream integrity


## Cell 2 — Upstream manifest và contract integrity

### Công việc

- xác minh manifest của Teacher A, Teacher B và Phase 03L;
- kiểm tra aggregate/registry hash của cả ba Teacher;
- xác minh Phase 2 shared training contract và decoder contract tồn tại;
- chỉ kiểm tra byte-level, không deserialize checkpoint.

### Đầu ra dự kiến

```text
Teacher A manifest        : ALL PASSED
Teacher B manifest        : ALL PASSED
Phase 03L manifest        : ALL PASSED
Teacher A/B/C KD qualified: True / True / True
Checkpoint payload read   : False
```


In [2]:
CELL_STARTED_AT = time.perf_counter()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def verify_manifest(root, csv_name, sums_name):
    csv_path = exactly_one(csv_name)
    sums_path = exactly_one(sums_name)
    assert csv_path.parent == sums_path.parent
    with csv_path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    expected = []
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file(), artifact
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"], artifact
        expected.append(f'{row["sha256"]}  {row["relative_path"]}')
    actual = [
        line.strip()
        for line in sums_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    assert actual == expected
    return len(rows), csv_path, sums_path


TEACHER_A_ROOT = TEACHER_A_COMPLETE.parent
TEACHER_B_ROOT = TEACHER_B_COMPLETE.parent
PHASE3L_ROOT = PHASE3_POOL_COMPLETE.parent
teacher_a_rows, teacher_a_manifest, teacher_a_sums = verify_manifest(
    TEACHER_A_ROOT,
    "phase3e_manifest.csv",
    "phase3e_SHA256SUMS.txt",
)
teacher_b_rows, teacher_b_manifest, teacher_b_sums = verify_manifest(
    TEACHER_B_ROOT,
    "phase3h_manifest.csv",
    "phase3h_SHA256SUMS.txt",
)
phase3l_rows, phase3l_manifest, phase3l_sums = verify_manifest(
    PHASE3L_ROOT,
    "phase3l_manifest.csv",
    "phase3l_SHA256SUMS.txt",
)
assert teacher_a_marker["qualified_for_kd"] is True
assert teacher_a_marker["quality_gate_passed_seeds"] == 3
assert teacher_b_marker["all_qualified_for_kd"] is True
assert teacher_b_marker["all_quality_gates_passed"] is True
assert phase3_pool_marker["all_teacher_runs_qualified_for_kd"] is True
assert phase3_pool_marker["all_teacher_quality_gates_passed"] is True

TEACHER_A_REGISTRY = resolve_inside(
    TEACHER_A_ROOT,
    teacher_a_marker["checkpoint_registry"]["relative_path"],
)
assert sha256_file(TEACHER_A_REGISTRY) == (
    teacher_a_marker["checkpoint_registry"]["sha256"]
)
TEACHER_B_AGGREGATE = resolve_inside(
    TEACHER_B_ROOT,
    teacher_b_marker["aggregate"]["relative_path"],
)
assert sha256_file(TEACHER_B_AGGREGATE) == (
    teacher_b_marker["aggregate"]["sha256"]
)
TEACHER_B_REGISTRY = resolve_inside(
    TEACHER_B_ROOT,
    teacher_b_marker["checkpoint_registry"]["relative_path"],
)
assert sha256_file(TEACHER_B_REGISTRY) == (
    teacher_b_marker["checkpoint_registry"]["sha256"]
)
TEACHER_C_AGGREGATE = resolve_inside(
    PHASE3L_ROOT,
    phase3_pool_marker["teacher_c_aggregate"]["relative_path"],
)
assert sha256_file(TEACHER_C_AGGREGATE) == (
    phase3_pool_marker["teacher_c_aggregate"]["sha256"]
)
TEACHER_C_REGISTRY = resolve_inside(
    PHASE3L_ROOT,
    phase3_pool_marker[
        "teacher_c_checkpoint_registry"
    ]["relative_path"],
)
assert sha256_file(TEACHER_C_REGISTRY) == (
    phase3_pool_marker[
        "teacher_c_checkpoint_registry"
    ]["sha256"]
)

PHASE2_TRAINING_CONTRACT = exactly_one(
    "phase2_shared_baseline_training_contract.json"
)
SHARED_DECODER_CONTRACT = exactly_one(
    "shared_transformer_decoder_contract.json"
)
phase2_training = read_json(PHASE2_TRAINING_CONTRACT)
decoder_contract = read_json(SHARED_DECODER_CONTRACT)
assert phase2_training["scope"]["test_set_status"] == "sealed"
assert int(decoder_contract["architecture"]["vocabulary_size"]) == 10080

write_json(
    AUDIT_DIR / "phase4a_cell02_upstream_integrity.json",
    {
        "audit_version": "phase4a_cell02_upstream_integrity_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "teacher_a_manifest_rows": teacher_a_rows,
        "teacher_b_manifest_rows": teacher_b_rows,
        "phase3l_manifest_rows": phase3l_rows,
        "teacher_a_registry_sha256": sha256_file(
            TEACHER_A_REGISTRY
        ),
        "teacher_b_aggregate_sha256": sha256_file(
            TEACHER_B_AGGREGATE
        ),
        "teacher_b_registry_sha256": sha256_file(
            TEACHER_B_REGISTRY
        ),
        "teacher_c_aggregate_sha256": sha256_file(
            TEACHER_C_AGGREGATE
        ),
        "teacher_c_registry_sha256": sha256_file(
            TEACHER_C_REGISTRY
        ),
        "phase2_training_contract_sha256": sha256_file(
            PHASE2_TRAINING_CONTRACT
        ),
        "shared_decoder_contract_sha256": sha256_file(
            SHARED_DECODER_CONTRACT
        ),
        "checkpoint_payload_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04A CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher A manifest rows   :", teacher_a_rows)
print("Teacher B manifest rows   :", teacher_b_rows)
print("Phase 03L manifest rows   :", phase3l_rows)
print("Teacher A manifest        : ALL PASSED")
print("Teacher B manifest        : ALL PASSED")
print("Phase 03L manifest        : ALL PASSED")
print("Teacher A/B/C KD qualified: True / True / True")
print("Shared vocabulary size    : 10080")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Next                      : build KD experiment matrix")


PHASE 04A CELL 2 STATUS
Status                    : PASSED
Teacher A manifest rows   : 12
Teacher B manifest rows   : 13
Phase 03L manifest rows   : 13
Teacher A manifest        : ALL PASSED
Teacher B manifest        : ALL PASSED
Phase 03L manifest        : ALL PASSED
Teacher A/B/C KD qualified: True / True / True
Shared vocabulary size    : 10080
Checkpoint payload read   : False
Test Set status           : SEALED
Next                      : build KD experiment matrix


## Cell 3 — KD experiment matrix và baseline targets

### Công việc

- đọc aggregate Validation của S1/S2/S3 và Teacher A/B/C;
- tạo ma trận cặp Teacher–Student và kênh KD hợp lệ;
- khóa cặp đầu tiên là Teacher A → S1 response/logit KD;
- không dùng hiệu năng Teacher để tuyên bố Student KD đã cải thiện.

### Đầu ra dự kiến

```text
Primary reproduction pair : Teacher A -> S1
Primary KD channel         : response/logit
Teacher B direct logit KD  : PROHIBITED
Teacher C direct logit KD  : ALLOWED
Held-out Test              : SEALED
```


In [3]:
CELL_STARTED_AT = time.perf_counter()
PHASE2_AGGREGATE_CSV = exactly_one("phase2_three_seed_aggregate.csv")
with PHASE2_AGGREGATE_CSV.open(
    "r", encoding="utf-8", newline=""
) as handle:
    baseline_rows = list(csv.DictReader(handle))
baseline_by_id = {
    row["student_id"]: row
    for row in baseline_rows
}
assert set(baseline_by_id) == {"S1", "S2", "S3"}

teacher_a_registry = read_json(TEACHER_A_REGISTRY)
teacher_b_aggregate = read_json(TEACHER_B_AGGREGATE)
teacher_c_aggregate = read_json(TEACHER_C_AGGREGATE)
teacher_c_registry = read_json(TEACHER_C_REGISTRY)
assert teacher_a_registry["status"] == "locked"
assert teacher_a_registry["qualified_for_kd"] is True
assert teacher_b_aggregate["status"] == "complete"
assert teacher_b_aggregate["all_qualified_for_kd"] is True
assert teacher_c_aggregate["status"] == "complete"
assert teacher_c_aggregate["all_qualified_for_kd"] is True
assert teacher_c_registry["status"] == "locked"
assert teacher_c_registry["entry_count"] == 3

teacher_a_cider = float(
    teacher_a_registry["aggregate_metrics"]["cider_d_mean"]
)
teacher_b_cider = float(
    teacher_b_aggregate["metrics"]["cider_d"]["mean"]
)
teacher_c_cider = float(
    teacher_c_aggregate["metrics"]["cider_d"]["mean"]
)
s1_cider = float(baseline_by_id["S1"]["cider_d_mean"])
s2_cider = float(baseline_by_id["S2"]["cider_d_mean"])
s3_cider = float(baseline_by_id["S3"]["cider_d_mean"])

KD_MATRIX = [
    {
        "experiment_id": "P4A_A_S1_LOGIT",
        "teacher_id": "A",
        "student_id": "S1",
        "channel": "response_logit",
        "status": "authorized_first",
        "compatibility": "shared decoder, vocabulary and token IDs",
        "teacher_validation_cider_d": teacher_a_cider,
        "student_baseline_validation_cider_d": s1_cider,
    },
    {
        "experiment_id": "P4A_A_S1_FEATURE",
        "teacher_id": "A",
        "student_id": "S1",
        "channel": "feature",
        "status": "after_projection_preflight",
        "compatibility": "requires explicit projection/layer contract",
        "teacher_validation_cider_d": teacher_a_cider,
        "student_baseline_validation_cider_d": s1_cider,
    },
    {
        "experiment_id": "P4A_A_S1_SEQUENCE",
        "teacher_id": "A",
        "student_id": "S1",
        "channel": "sequence",
        "status": "registered_after_reproduction",
        "compatibility": "shared deterministic decoding contract",
        "teacher_validation_cider_d": teacher_a_cider,
        "student_baseline_validation_cider_d": s1_cider,
    },
    {
        "experiment_id": "P4A_B_S2_SEQUENCE",
        "teacher_id": "B",
        "student_id": "S2",
        "channel": "sequence",
        "status": "authorized_after_pseudo_caption_contract",
        "compatibility": "native BLIP captions; no token-logit alignment",
        "teacher_validation_cider_d": teacher_b_cider,
        "student_baseline_validation_cider_d": s2_cider,
    },
    {
        "experiment_id": "P4A_B_S2_FEATURE",
        "teacher_id": "B",
        "student_id": "S2",
        "channel": "feature",
        "status": "after_projection_preflight",
        "compatibility": "requires explicit projection/layer contract",
        "teacher_validation_cider_d": teacher_b_cider,
        "student_baseline_validation_cider_d": s2_cider,
    },
    {
        "experiment_id": "P4A_B_S2_LOGIT",
        "teacher_id": "B",
        "student_id": "S2",
        "channel": "response_logit",
        "status": "prohibited",
        "compatibility": "BLIP vocabulary is not Student-index compatible",
        "teacher_validation_cider_d": teacher_b_cider,
        "student_baseline_validation_cider_d": s2_cider,
    },
    {
        "experiment_id": "P4A_C_S3_LOGIT",
        "teacher_id": "C",
        "student_id": "S3",
        "channel": "response_logit",
        "status": "registered_after_a_to_s1_reproduction",
        "compatibility": "shared decoder, vocabulary and token IDs",
        "teacher_validation_cider_d": teacher_c_cider,
        "student_baseline_validation_cider_d": s3_cider,
    },
    {
        "experiment_id": "P4A_C_S3_FEATURE",
        "teacher_id": "C",
        "student_id": "S3",
        "channel": "feature",
        "status": "after_spatial_alignment_preflight",
        "compatibility": "align Teacher 196 tokens to Student 49 tokens",
        "teacher_validation_cider_d": teacher_c_cider,
        "student_baseline_validation_cider_d": s3_cider,
    },
    {
        "experiment_id": "P4A_C_S3_SEQUENCE",
        "teacher_id": "C",
        "student_id": "S3",
        "channel": "sequence",
        "status": "registered_after_reproduction",
        "compatibility": "shared deterministic decoding contract",
        "teacher_validation_cider_d": teacher_c_cider,
        "student_baseline_validation_cider_d": s3_cider,
    },
]

MATRIX_CSV = RESULT_DIR / "phase4_kd_experiment_matrix.csv"
with MATRIX_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(KD_MATRIX[0]))
    writer.writeheader()
    writer.writerows(KD_MATRIX)

BASELINE_TARGETS = RESULT_DIR / "phase4_baseline_targets.json"
write_json(
    BASELINE_TARGETS,
    {
        "schema_version": "phase4_baseline_targets_v1",
        "created_at_utc": now_utc(),
        "selection_split": "validation",
        "primary_metric": "CIDEr-D",
        "students": {
            student_id: {
                "name": row["student_name"],
                "cider_d_mean": float(row["cider_d_mean"]),
                "cider_d_sample_sd": float(
                    row["cider_d_sample_sd"]
                ),
                "bleu_4_mean": float(row["bleu_4_mean"]),
                "validation_ce_mean": float(
                    row["validation_ce_mean"]
                ),
            }
            for student_id, row in baseline_by_id.items()
        },
        "teachers": {
            "A": {
                "cider_d_mean": teacher_a_cider,
                "qualified_for_kd": True,
            },
            "B": {
                "cider_d_mean": teacher_b_cider,
                "qualified_for_kd": True,
                "direct_logit_kd": "prohibited",
            },
            "C": {
                "cider_d_mean": teacher_c_cider,
                "qualified_for_kd": True,
                "direct_logit_kd": "allowed",
                "feature_alignment": "196_to_49_visual_tokens",
            },
        },
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

write_json(
    AUDIT_DIR / "phase4a_cell03_experiment_matrix.json",
    {
        "audit_version": "phase4a_cell03_experiment_matrix_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "experiment_count": len(KD_MATRIX),
        "primary_experiment": "P4A_A_S1_LOGIT",
        "s1_baseline_cider_d": s1_cider,
        "s2_baseline_cider_d": s2_cider,
        "s3_baseline_cider_d": s3_cider,
        "teacher_a_cider_d": teacher_a_cider,
        "teacher_b_cider_d": teacher_b_cider,
        "teacher_c_cider_d": teacher_c_cider,
        "teacher_b_direct_logit_kd": "prohibited",
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04A CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Primary reproduction pair : Teacher A -> S1")
print("Primary KD channel        : response/logit")
print("S1 baseline CIDEr-D       :", round(s1_cider, 6))
print("Teacher A CIDEr-D         :", round(teacher_a_cider, 6))
print("Teacher B sequence KD     : REGISTERED")
print("Teacher B direct logit KD : PROHIBITED")
print("Teacher C logit KD        : REGISTERED")
print("Teacher C feature KD      : REQUIRES 196 -> 49 ALIGNMENT")
print("Held-out Test             : SEALED")
print("Matrix                    :", MATRIX_CSV)
print("Next                      : lock loss contract")


PHASE 04A CELL 3 STATUS
Status                    : PASSED
Primary reproduction pair : Teacher A -> S1
Primary KD channel        : response/logit
S1 baseline CIDEr-D       : 0.985831
Teacher A CIDEr-D         : 1.160464
Teacher B sequence KD     : REGISTERED
Teacher B direct logit KD : PROHIBITED
Teacher C logit KD        : REGISTERED
Teacher C feature KD      : REQUIRES 196 -> 49 ALIGNMENT
Held-out Test             : SEALED
Matrix                    : /kaggle/working//phase4_kd_reproduction_entry/results/phase4_kd_experiment_matrix.csv
Next                      : lock loss contract


## Cell 4 — Response/logit KD loss contract cho reproduction đầu tiên

### Công việc

- khóa công thức loss tái lập đầu tiên;
- giữ CE token loss và thêm KL divergence trên vocabulary chung;
- khóa temperature, alpha và padding mask;
- định nghĩa gate để chứng minh implementation đúng trước full training.

### Loss

```text
L = (1 - alpha) * CE + alpha * T^2 * KL(pTeacher(T) || pStudent(T))
```

### Cấu hình đầu tiên

- `T = 4.0`
- `alpha = 0.5`
- chỉ tính trên token không phải PAD;
- Teacher ở `eval()` và không nhận gradient;
- Student nhận gradient từ CE và KD.

Đây là cấu hình reproduction cố định, không phải kết quả tuning trên Test.
Các giá trị `T` và `alpha` chỉ được thay đổi ở Phase 5 ablation/cải tiến.


In [4]:
CELL_STARTED_AT = time.perf_counter()
KD_LOSS_CONTRACT = CONFIG_DIR / (
    "teacher_a_s1_response_logit_kd_loss_contract.json"
)
loss_contract = {
    "contract_version": "phase4_teacher_a_s1_logit_kd_loss_v1",
    "created_at_utc": now_utc(),
    "status": "locked_for_preflight",
    "experiment_id": "P4A_A_S1_LOGIT",
    "teacher": {
        "teacher_id": "A",
        "architecture": (
            "CLIP ViT-B/32 + visual adapter + "
            "shared caption decoder"
        ),
        "mode": "eval",
        "requires_grad": False,
    },
    "student": {
        "student_id": "S1",
        "architecture": "MobileNetV3-Large + shared decoder",
        "requires_grad": True,
    },
    "compatibility": {
        "vocabulary_size": 10080,
        "shared_token_ids": True,
        "shared_decoder_contract": True,
        "pad_id": 0,
        "direct_logit_kd_allowed": True,
    },
    "loss": {
        "formula": (
            "(1-alpha)*CE + alpha*T^2*KL("
            "teacher_soft_targets || student_log_probs)"
        ),
        "temperature": 4.0,
        "alpha": 0.5,
        "ce_weight": 0.5,
        "kd_weight_before_temperature_scale": 0.5,
        "temperature_squared_scale": True,
        "reduction": "sum over valid tokens / valid token count",
        "padding_mask": "labels != PAD_ID",
        "teacher_logits_detached": True,
    },
    "preflight_gates": [
        "teacher and student logits shape identical",
        "teacher parameters receive no gradient",
        "student gradients finite",
        "CE, KL and combined loss finite",
        "alpha=0 reproduces CE-only loss",
        "identical logits produce near-zero KL",
        "one-batch overfit passes",
        "exact resume passes",
    ],
    "selection_split": "validation",
    "primary_metric": "CIDEr-D",
    "test_set_status": "sealed",
    "test_set_used": False,
    "phase3_authorization": {
        "path": str(PHASE4_AUTHORIZED),
        "sha256": sha256_file(PHASE4_AUTHORIZED),
    },
}
write_json(KD_LOSS_CONTRACT, loss_contract)

write_json(
    AUDIT_DIR / "phase4a_cell04_loss_contract.json",
    {
        "audit_version": "phase4a_cell04_loss_contract_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "loss_contract": str(KD_LOSS_CONTRACT),
        "loss_contract_sha256": sha256_file(KD_LOSS_CONTRACT),
        "temperature": 4.0,
        "alpha": 0.5,
        "teacher_frozen": True,
        "shared_vocabulary_verified": True,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04A CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("KD type                   : response/logit")
print("Temperature               : 4.0")
print("Alpha                     : 0.5")
print("Vocabulary size           : 10080")
print("PAD masking               : REQUIRED")
print("Teacher gradient           : DISABLED")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Contract                  :", KD_LOSS_CONTRACT)
print("Next                      : lock training and evaluation policy")


PHASE 04A CELL 4 STATUS
Status                    : PASSED
KD type                   : response/logit
Temperature               : 4.0
Alpha                     : 0.5
Vocabulary size           : 10080
PAD masking               : REQUIRED
Teacher gradient           : DISABLED
Training started          : False
Test Set status           : SEALED
Contract                  : /kaggle/working//phase4_kd_reproduction_entry/configs/teacher_a_s1_response_logit_kd_loss_contract.json
Next                      : lock training and evaluation policy


## Cell 5 — Training, evaluation và confirmation policy

### Công việc

- kế thừa Student baseline training budget để so sánh công bằng;
- chạy seed 42 trước, sau đó xác nhận seed 123 và 2026;
- checkpoint selection chỉ dùng Validation CIDEr-D;
- định nghĩa điều kiện thành công so với đúng S1 CE-only baseline.

### Đầu ra dự kiến

```text
Development seed          : 42
Confirmation seeds        : [123, 2026]
Selection split           : Validation
Primary metric            : CIDEr-D
Held-out Test             : SEALED
```


In [5]:
CELL_STARTED_AT = time.perf_counter()
TRAINING_POLICY = CONFIG_DIR / (
    "phase4_teacher_a_s1_logit_kd_training_policy.json"
)
training_policy = {
    "contract_version": "phase4_teacher_a_s1_logit_kd_training_v1",
    "created_at_utc": now_utc(),
    "status": "locked_for_preflight",
    "experiment_id": "P4A_A_S1_LOGIT",
    "baseline_contract": {
        "path": str(PHASE2_TRAINING_CONTRACT),
        "sha256": sha256_file(PHASE2_TRAINING_CONTRACT),
    },
    "fair_comparison": {
        "dataset": "MS COCO 2014 only",
        "split_protocol": "Karpathy",
        "training_images_per_epoch": 113287,
        "caption_sampling": "one caption per image per epoch, cyclic",
        "physical_batch_size": 32,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 64,
        "maximum_epochs": 20,
        "optimizer": "AdamW",
        "backbone_lr": 1e-4,
        "adapter_lr": 3e-4,
        "decoder_lr": 3e-4,
        "weight_decay": 1e-4,
        "scheduler": "warmup 5 percent plus cosine",
        "mixed_precision": "FP16 plus GradScaler",
        "gradient_clip_norm": 1.0,
        "generation": "Beam-3, length penalty 0.7",
    },
    "seeds": {
        "development": 42,
        "confirmation": [123, 2026],
    },
    "selection": {
        "split": "validation",
        "primary_metric": "CIDEr-D",
        "tie_break": ["BLEU-4", "validation CE", "earlier epoch"],
        "early_stopping": {
            "minimum_epoch": 5,
            "patience": 4,
            "minimum_delta": 0.001,
        },
    },
    "success_reporting": {
        "primary_comparison": (
            "KD Student S1 versus CE-only Student S1 under matched seed"
        ),
        "aggregate_required": True,
        "sample_standard_deviation_required": True,
        "teacher_metric_not_counted_as_kd_improvement": True,
        "compute_reporting": [
            "parameters",
            "training time",
            "peak GPU memory",
            "generation time",
        ],
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(TRAINING_POLICY, training_policy)

write_json(
    AUDIT_DIR / "phase4a_cell05_training_policy.json",
    {
        "audit_version": "phase4a_cell05_training_policy_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "development_seed": 42,
        "confirmation_seeds": [123, 2026],
        "selection_split": "validation",
        "primary_metric": "CIDEr-D",
        "matched_ce_only_baseline_required": True,
        "teacher_metric_is_not_kd_result": True,
        "phase3_authorization_verified": True,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "phase3_complete_anchor": {
            "path": str(PHASE3_POOL_COMPLETE),
            "sha256": sha256_file(PHASE3_POOL_COMPLETE),
        },
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04A CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Development seed          : 42")
print("Confirmation seeds        : [123, 2026]")
print("Selection split           : Validation")
print("Primary metric            : CIDEr-D")
print("Matched baseline          : S1 CE-only")
print("Teacher metric as KD gain : PROHIBITED")
print("Training started          : False")
print("Held-out Test             : SEALED")
print("Policy                    :", TRAINING_POLICY)
print("Next                      : package Phase 04A")


PHASE 04A CELL 5 STATUS
Status                    : PASSED
Development seed          : 42
Confirmation seeds        : [123, 2026]
Selection split           : Validation
Primary metric            : CIDEr-D
Matched baseline          : S1 CE-only
Teacher metric as KD gain : PROHIBITED
Training started          : False
Held-out Test             : SEALED
Policy                    : /kaggle/working//phase4_kd_reproduction_entry/configs/phase4_teacher_a_s1_logit_kd_training_policy.json
Next                      : package Phase 04A


## Cell 6 — Package Phase 04A và authorize Phase 04B

### Công việc

- tạo manifest CSV/JSON/SHA256SUMS;
- tạo Phase 04A completion marker;
- authorize notebook Phase 04B Teacher A → S1 logit KD preflight;
- không sao chép checkpoint lớn vào output.

### Đầu ra dự kiến

```text
Status                    : COMPLETE
Primary pair              : Teacher A -> S1
KD channel                : response/logit
Phase 04B preflight       : AUTHORIZED
Training started          : False
Test Set status           : SEALED
```


In [6]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase4a_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase4a_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase4a_SHA256SUMS.txt"
COMPLETE_MARKER = OUTPUT_ROOT / (
    "PHASE4_KD_REPRODUCTION_ENTRY_COMPLETE.json"
)
AUTHORIZED_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_A_S1_LOGIT_KD_PREFLIGHT_AUTHORIZED.json"
)

write_json(
    AUTHORIZED_MARKER,
    {
        "marker_version": (
            "phase4_teacher_a_s1_logit_kd_preflight_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "authorized",
        "experiment_id": "P4A_A_S1_LOGIT",
        "teacher_id": "A",
        "student_id": "S1",
        "kd_channel": "response_logit",
        "loss_contract": {
            "relative_path": KD_LOSS_CONTRACT.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(KD_LOSS_CONTRACT),
        },
        "training_policy": {
            "relative_path": TRAINING_POLICY.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(TRAINING_POLICY),
        },
        "next_phase": (
            "Phase 04B Teacher A to S1 logit KD preflight"
        ),
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "phase3_complete_sha256": sha256_file(
            PHASE3_POOL_COMPLETE
        ),
        "phase4_upstream_authorization_sha256": sha256_file(
            PHASE4_AUTHORIZED
        ),
        "matched_pair_count": 3,
        "registered_experiment_count": len(KD_MATRIX),
    },
)

artifact_paths = sorted(
    [
        path
        for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV,
            MANIFEST_JSON,
            SHA256SUMS,
            COMPLETE_MARKER,
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase4a_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)

for row in manifest_rows:
    path = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert path.stat().st_size == int(row["size_bytes"])
    assert sha256_file(path) == row["sha256"]

write_json(
    COMPLETE_MARKER,
    {
        "marker_version": "phase4_kd_reproduction_entry_complete_v1",
        "created_at_utc": now_utc(),
        "status": "complete",
        "primary_experiment": "P4A_A_S1_LOGIT",
        "teacher_id": "A",
        "student_id": "S1",
        "kd_channel": "response_logit",
        "temperature": 4.0,
        "alpha": 0.5,
        "phase04b_preflight_authorized": True,
        "authorization_marker": {
            "relative_path": AUTHORIZED_MARKER.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(AUTHORIZED_MARKER),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
            },
            "json": {
                "relative_path": MANIFEST_JSON.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_JSON),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04A CELL 6 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Primary pair              : Teacher A -> S1")
print("Matched pairs locked      : 3 / 3")
print("Registered KD channels    :", len(KD_MATRIX))
print("KD channel                : response/logit")
print("Temperature / alpha       : 4.0 / 0.5")
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Phase 04B preflight       : AUTHORIZED")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Complete marker           :", COMPLETE_MARKER)
print(
    "Elapsed                   :",
    round(time.perf_counter() - CELL_STARTED_AT, 2),
    "giây",
)
print("Next                      : Save Version, then run Phase 04B")


PHASE 04A CELL 6 STATUS
Status                    : COMPLETE
Primary pair              : Teacher A -> S1
Matched pairs locked      : 3 / 3
Registered KD channels    : 9
KD channel                : response/logit
Temperature / alpha       : 4.0 / 0.5
Manifest rows             : 10
Manifest verification     : ALL PASSED
Phase 04B preflight       : AUTHORIZED
Training started          : False
Test Set status           : SEALED
Complete marker           : /kaggle/working//phase4_kd_reproduction_entry/PHASE4_KD_REPRODUCTION_ENTRY_COMPLETE.json
Elapsed                   : 0.02 giây
Next                      : Save Version, then run Phase 04B
